In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as dt
from typing import Union, Optional

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
## Define Today Date

today_date = pd.to_datetime('2022-01-01')

In [ ]:
## Import Data

df_or = pd.read_csv('/kaggle/input/wholesale-and-retail-orders-dataset/orders.csv')
df_sp = pd.read_csv('/kaggle/input/wholesale-and-retail-orders-dataset/product-supplier.csv')

### DataFrame Checking

In [ ]:
df_or.head()

In [ ]:
df_sp.head()

In [ ]:
df_or.info()

In [ ]:
df_sp.info()

#### *Initial Check Note:*
  - Columns name formating issue
  - Customer Status Values formating issue
  - Merging
------------------

### DataFrame Cleaning

In [ ]:
df_or.columns

In [ ]:
## Change columns name format

df_or = df_or.rename(columns = {'Customer ID': 'customer_id',
                                'Customer Status': 'customer_tier',
                                'Date Order was placed': 'order_date',
                                'Delivery Date': 'delivery_date',
                                'Order ID': 'order_id',
                                'Product ID': 'product_id',
                                'Quantity Ordered': 'order_qty',
                                'Total Retail Price for This Order': 'total_bill',
                                'Cost Price Per Unit': 'cost_per_unit'})

In [ ]:
# Change customer status values format

df_or['customer_tier'] = df_or['customer_tier'].replace(['SILVER'], 'silver')
df_or['customer_tier'] = df_or['customer_tier'].replace(['GOLD'], 'gold')
df_or['customer_tier'] = df_or['customer_tier'].replace(['PLATINUM'], 'platinum')
df_or['customer_tier'] = df_or['customer_tier'].replace(['Silver'], 'silver')
df_or['customer_tier'] = df_or['customer_tier'].replace(['Gold'], 'gold')
df_or['customer_tier'] = df_or['customer_tier'].replace(['Platinum'], 'platinum')

In [ ]:
df_sp.columns

In [ ]:
# Change columns name format

df_sp = df_sp.rename(columns = {'Product ID': 'product_id',
                                'Product Line': 'product_line',
                                'Product Category': 'product_category',
                                'Product Group': 'product_group',
                                'Product Name': 'product_name',
                                'Supplier Country': 'supplier_country',
                                'Supplier Name': 'supplier_name',
                                'Supplier ID': 'supplier_id'})

In [ ]:
df = pd.merge(df_or, df_sp, how = 'inner', on = 'product_id')
df.head()

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

df['fulfillment_time'] = (df['delivery_date']) - (df['order_date'])
df['fulfillment_time'] = df['fulfillment_time'].dt.days

df['total_cost'] = (df['order_qty']) * (df['cost_per_unit'])
df['profit'] = (df['total_bill']) - (df['total_cost'])
df['margin'] = (((df['total_bill']) - (df['total_cost'])) 
                   / (df['total_bill'])) * 100

df.head()

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

In [ ]:
df.columns

In [ ]:
df = df[['customer_id', 'customer_tier', 'order_id', 'order_date', 'order_qty', 
         'cost_per_unit', 'product_id', 'product_line', 'product_category', 
         'product_group', 'product_name', 'supplier_id','supplier_country',
         'supplier_name', 'delivery_date', 'fulfillment_time', 'total_cost', 
         'total_bill','profit', 'margin']]

### 1. The "Tier vs. Margin" Leakage Analysis

In [ ]:
tivmr = df.copy()
tivmr.head()

In [ ]:
print('OA Avg Profit:', tivmr['profit'].mean())
print('Pl Avg Profit:', tivmr.loc[tivmr['customer_tier']=='platinum', 
      'profit'].mean())
print('Gl Avg Profit:', tivmr.loc[tivmr['customer_tier']=='gold', 
      'profit'].mean())
print('Sl Avg Profit:', tivmr.loc[tivmr['customer_tier']=='silver', 
      'profit'].mean())
print('------')
print('OA Avg Margin:', tivmr['margin'].mean())
print('Pl Avg Margin:', tivmr.loc[tivmr['customer_tier']=='platinum', 
      'margin'].mean())
print('Gl Avg Margin:', tivmr.loc[tivmr['customer_tier']=='gold', 
      'margin'].mean())
print('Sl Avg Margin:', tivmr.loc[tivmr['customer_tier']=='silver', 
      'margin'].mean())

In [ ]:
# Create a boxplot to visualize distribution of 'profit' and 'margin'

fig, (ax1, ax2) = plt.subplots(2, 1 , figsize = (12, 12))

sns.boxplot(data = tivmr, x = 'profit', y = 'customer_tier', hue = 'customer_tier', ax = ax1)
ax1.set_xlim([20,92])
ax1.axvline(x = 73.6078653662175, color = 'r', ls = '--')
ax1.axvline(x = 73.20456591676296, color = 'orange', ls = '--')
ax1.axvline(x = 73.97386639435494, color = 'b', ls = '--')
ax1.axvline(x = 74.02092393896042, color = 'g', ls = '--')
ax1.set_ylabel('Customer Tier')
ax1.set_xlabel('Dollar Profit')
ax1.set_title('Average Profit per Customer Tier')

sns.boxplot(data = tivmr, x = 'margin', y = 'customer_tier', hue = 'customer_tier', ax = ax2)
ax2.axvline(x = 53.895403372773835, color = 'r', ls = '--')
ax2.set_ylabel('Customer Tier')
ax2.set_xlabel('Margin Percentage')
ax2.set_title('Average Profit Margin per Customer Tier');

In [ ]:
fig, ax1 = plt.subplots(figsize = (12, 8))
ax2 = ax1.twinx()
sns.barplot(data = tivmr, x = 'customer_tier', y = 'total_bill', 
            estimator = 'mean', hue = 'customer_tier', ax = ax1)
ax1.set_xlabel('Customer Tier')
ax1.set_ylabel('Total Bill')
ax1.set_title('Average Revenue vs Order Quantity Across Customer Tier ')
sns.lineplot(data = tivmr, x = 'customer_tier', y = 'order_qty', marker = 'o',
             color = 'r', estimator = 'mean', ax = ax2)
ax2.set_ylabel('Order Quantity');

### 2. Oder Fullfilment Latency Analysis

In [ ]:
lt = df.copy()

In [ ]:
print('OA fulfillment Time:', lt['fulfillment_time'].mean())

print('----------')

print('US Avg fulfillment_time:', lt.loc[lt['supplier_country']=='US', 
      'fulfillment_time'].mean())
print('GB Avg fulfillment_time:', lt.loc[lt['supplier_country']=='GB', 
      'fulfillment_time'].mean())
print('NL Avg fulfillment_time:', lt.loc[lt['supplier_country']=='NL', 
      'fulfillment_time'].mean())
print('PT Avg fulfillment_time:', lt.loc[lt['supplier_country']=='PT', 
      'fulfillment_time'].mean())
print('NO Avg fulfillment_time:', lt.loc[lt['supplier_country']=='NO', 
      'fulfillment_time'].mean())
print('ES Avg fulfillment_time:', lt.loc[lt['supplier_country']=='ES', 
      'fulfillment_time'].mean())
print('BE Avg fulfillment_time:', lt.loc[lt['supplier_country']=='BE', 
      'fulfillment_time'].mean())
print('CA Avg fulfillment_time:', lt.loc[lt['supplier_country']=='CA', 
      'fulfillment_time'].mean())
print('AU Avg fulfillment_time:', lt.loc[lt['supplier_country']=='AU', 
      'fulfillment_time'].mean())
print('FR Avg fulfillment_time:', lt.loc[lt['supplier_country']=='FR', 
      'fulfillment_time'].mean())
print('SE Avg fulfillment_time:', lt.loc[lt['supplier_country']=='SE', 
      'fulfillment_time'].mean())
print('DK Avg fulfillment_time:', lt.loc[lt['supplier_country']=='DK', 
      'fulfillment_time'].mean())
print('DE Avg fulfillment_time:', lt.loc[lt['supplier_country']=='DE', 
      'fulfillment_time'].mean())

print('----------')

print('Pl Avg fulfillment_time:', lt.loc[lt['customer_tier']=='platinum', 
      'fulfillment_time'].mean())
print('Gl Avg fulfillment_time:', lt.loc[lt['customer_tier']=='gold', 
      'fulfillment_time'].mean())
print('Sl Avg fulfillment_time:', lt.loc[lt['customer_tier']=='silver', 
      'fulfillment_time'].mean())

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2 , figsize = (15, 8))

sns.barplot(data = lt, x = 'supplier_country', y = 'fulfillment_time', 
            estimator = 'mean', hue = 'supplier_country', ax = ax1)
ax1.axhline(y = 1.056806818980288, color = 'r', ls = '--')
ax1.set_xlabel('Supplier Origin')
ax1.set_ylabel('Fulfillment Time')
ax1.set_title('Average Fulfillment Time by Supplier Country Origin')

sns.barplot(data = lt, x = 'customer_tier', y = 'fulfillment_time', hue = 'customer_tier',
            estimator = 'mean', ax = ax2)
ax2.axhline(y = 1.056806818980288, color = 'r', ls = '--')
ax2.set_xlabel('Customer Tier')
ax2.set_ylabel('Fulfillment Time')
ax2.set_title('Average Fulfillment Time per Customer Tier');

### 3. Product Portfolio Performance

#### a. Losing Products

In [ ]:
lost = df[df['profit']<0].copy()

lost['month'] = lost['order_date'].dt.month

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2 , figsize = (15,6))

sns.barplot(data = lost,  x = 'customer_tier', y = 'order_qty', estimator = 'sum',
            palette = ['orange', 'b','g'], ax = ax1)
ax1.set_xlabel('Customer Tier')
ax1.set_ylabel('Order Quantity')

sns.barplot(data = lost, x = 'customer_tier', y = 'profit', estimator = 'sum', 
            palette = ['orange', 'b','g'], ax = ax2)
ax2.set_xlabel('Customer Tier')
ax2.set_ylabel('Profit')
ax2.invert_yaxis()

fig.suptitle('Customer Tier Distribution on Losing Product');

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3 , figsize = (19,6))

sns.barplot(data = lost,  x = 'product_name', y = 'order_qty', estimator = 'sum', hue = 'product_name',
            ax = ax1)
ax1.tick_params(axis = 'x', rotation = 90)
ax1.set_xlabel('Order Quantity')
ax1.set_ylabel('Product')

sns.barplot(data = lost, x = 'product_name', y = 'profit', estimator = 'sum', hue = 'product_name',
            ax = ax2)
ax2.invert_yaxis()
ax2.tick_params(axis = 'x', rotation = 90)
ax2.set_xlabel('Profit')
ax2.set_ylabel('Product')

sns.barplot(data = lost,  x = 'month', y = 'order_qty', estimator = 'sum', hue = 'month',
            ax = ax3)
ax3.set_xlabel('Order Quantity')
ax3.set_ylabel('Month')

fig.suptitle('Losing Product');

In [ ]:
print('FHP Total Profit:', df.loc[df['product_name'] == 'Football - Helmet Pro XL', 
      'profit'].sum())
print('HLL Total Profit:', df.loc[df['product_name'] == 'Helmet L', 
      'profit'].sum())
print('FSB Total Profit:', df.loc[df['product_name'] == 'Football Super Bowl', 
      'profit'].sum())
print('WRS Total Profit:', df.loc[df['product_name'] == "Perfect Fit Women's Roller Skates XP",
      'profit'].sum())
print('NDJ Total Profit:', df.loc[df['product_name'] == 'Nova Down Jacket', 
      'profit'].sum())

print('----------')

print('FHP Total Profit:', lost.loc[lost['product_name'] == 'Football - Helmet Pro XL', 
      'profit'].sum())
print('HLL Total Profit:', lost.loc[lost['product_name'] == 'Helmet L', 
      'profit'].sum())
print('FSB Total Profit:', lost.loc[lost['product_name'] == 'Football Super Bowl', 
      'profit'].sum())
print('WRS Total Profit:', lost.loc[lost['product_name'] == "Perfect Fit Women's Roller Skates XP", 
      'profit'].sum())
print('NDJ Total Profit:', lost.loc[lost['product_name'] == 'Nova Down Jacket', 
      'profit'].sum())

#### b. Winning Product

In [ ]:
pr = df.copy()
pr.head()

In [ ]:
fig, ax1 = plt.subplots(figsize = (12, 8))

ax2 = ax1.twinx()

sns.barplot(data = pr, x = 'product_category', y = 'profit', hue = 'product_category',
           estimator = 'sum', ax = ax1)
ax1.tick_params(axis = 'x', rotation = 90)
ax1.set_xlabel('Product Category')
ax1.set_ylabel('Profit')

sns.lineplot(data = pr, x = 'product_category', y = 'order_qty', marker = 'o',
             color = 'r', estimator = 'sum', ax = ax2)
ax2.set_ylabel('Order Quantity')

fig.suptitle('Total Profit per Product Category vs the Order Quantity');

In [ ]:
catfil = ['Outdoors', 'Assorted Sports Articles', 'Clothes', 'Shoes', 'Winter Sports']
winn = pr[pr['product_category'].isin(catfil)]

In [ ]:
fig, ax1 = plt.subplots(figsize = (12, 8))

ax2 = ax1.twinx()

sns.barplot(data = winn, x = 'product_group', y = 'profit', hue = 'product_group',
           estimator = 'sum', ax = ax1)
ax1.set_xlabel('Product Group')
ax1.set_ylabel('Profit')
ax1.tick_params(axis = 'x', rotation = 90)

sns.lineplot(data = winn, x = 'product_group', y = 'order_qty', marker = 'o',
             color = 'r', estimator = 'sum', ax = ax2)
ax2.set_ylabel('Order Quantity')

fig.suptitle('Total Profit per Product Group vs the Order Quantity');

In [ ]:
grfil = ['Skates','Eclipse Shoes', 'Assorted Sports Articles', 'Tents', 'Anoraks & Parkas']
winna = winn[winn['product_group'].isin(grfil)]

In [ ]:
fig, ax1 = plt.subplots(figsize = (50, 15))

ax2 = ax1.twinx()

sns.barplot(data = winna, x = 'product_name', y = 'profit', hue = 'product_name',
           estimator = 'sum', ax = ax1)
ax1.set_xlabel('Product Name')
ax1.set_ylabel('Profit')
ax1.tick_params(axis = 'x', rotation = 90)

sns.lineplot(data = winna, x = 'product_name', y = 'order_qty', marker = 'o',
             color = 'r', estimator = 'sum', ax = ax2)
ax2.set_ylabel('Order Quantity')

fig.suptitle('Total Profit per Product vs the Order Quantity');

In [ ]:
print('FH4 Total Profit:', df.loc[df['product_name'] == 'Family Holiday 4', 
      'profit'].sum())
print('HR4 Total Profit:', df.loc[df['product_name'] == 'Hurricane 4', 
      'profit'].sum())
print('FH6 Total Profit:', df.loc[df['product_name'] == 'Family Holiday 6',
      'profit'].sum())
print('ED3 Total Profit:', df.loc[df['product_name'] == 'Expedition Dome 3', 
      'profit'].sum())
print('CMS Total Profit:', df.loc[df['product_name'] == 'Comfort Shelter', 
      'profit'].sum())

-----

## 1. Customer Tier Analysis: "Myth-Busting" the Platinum Tier

We conducted an analysis to determine if Platinum discounts are eroding our margins or if they represent a "Cash Cow" segment.

**Key Findings:**

| Tier | Avg.Profit per Order | Avg. Profit Margin (%) | Performance Note |
|---|---|---|---|
| Platinum | 74.02 | 53.88% | Highest revenue per transaction |
| Gold | 73.20 | 53.89% | Consistent with baseline |
| Silver | 73.97 | 53.89% | Consistent with baseline |
| **Average** | **73.61** | **53.90%** | **Benchmark** |

* **Uniform Profitability**: There is no significant variance in profit margins across tiers. All three tiers (Platinum, Gold, Silver) maintain a stable average profit margin of approximately 53.9%.

* **Discount Integrity**: The data disproves the assumption that we are over-discounting for Platinum members. Our pricing logic is holding firm across the board.

* **Revenue Efficiency**: While Platinum members show slightly lower purchasing volume, they generate higher revenue per order. However, they do not yet meet the "Cash Cow" criteria of significantly higher margins compared to other segments.

**Strategic Takeaway**: *Our loyalty tiers are stable, but we are not currently capturing "premium" margins from our highest-tier customers.*

---
## 2. Operational & Fulfillment Performance

We evaluated fulfillment speeds across international suppliers and customer tiers to identify service-level gaps.

**Key Findings:**

* **Fulfillment Speed by Tier**: Analysis shows that our fulfillment speed is **slowest** for our most valuable customers.

|Customer Tier|Avg. Orders Fulfilment (days)|Status|
|---|---|---|
| Platinum |1.077|**Slowest**| 
| Gold |1.052|Fastest|
| Silver |1.060|Average|

**The "Service Gap" Risk**: *Currently, there is a negative incentive for tier advancement. Platinum members—who contribute the highest revenue per order—are waiting longer for their orders to be fulfilled than Gold or Silver members.*

* **Supplier Bottleneck**: Products sourced from Germany (DE) exhibit the longest warehouse-to-fulfillment latency. This is a primary area for logistics optimization.

|Supplier Origin|Avg. Fulfilment Orders (days)|
|---|---|
| US |1.002| 
| GB |1.113| 
| NL |0.998| 
| PT |1.264| 
| NO |1.021|
| ES |1.137| 
| BE |1.130| 
| CA |1.107| 
| AU |1.095| 
| FR |1.105|
| SE |1.025| 
| DK |1.245| 
| DE |1.554|

---
## 3. Product Performance & "Red Flag" Audit

We identified our primary profit drivers and isolated 54 specific transactions where costs exceeded retail prices.

**Key Findings:**

* **Top 5 Profit Drivers**

Our partnership with **Petterson AB (Sweden)** remains our most critical revenue engine.

| Product Name | Supplier Origin | Supplier Name  | All-Time Profit |
|---|---|---|---|
| Family Holiday 4 | SE | Petterson AB | 282460.5 |
| Hurricane 4 | SE | Petterson AB | 194804.2 |
| Family Holiday 6 | SE | Petterson AB | 139843.2 |
| Expedition Dome 3 | GB | Outback Outfitters Ltd | 123326.0 |
| Comfort Shelter | GB | Outback Outfitters Ltd | 106420.0 |

* **The "Red Flag" List (Margin Leakage)**

We isolated 54 orders where the company realized a net loss. This "Profit Erosion" is most visible in our **team sports category**.

| Product Name | All-Time Profit  | Sum of Negative Margin Orders | Profit Erosion % |
|---|---|---|---|
| Football - Helmet Pro XL | 6831.64 | -190.80 | 2.8% |
| Helmet L | 1583.89 | -114.66 | 7.2% |
| Football Super Bowl | 3498.66 | -33.64 | 0.9 |
| Perfect Fit Women's Roller Skates XP | 36717.41 | -24.57 | 0.07 |
| Nova Down Jacket | 0.0 | 0.0 | N/A |

**Observation**:
*We have identified a 7.2% profit erosion on **Baseball Helmet L** item. Because this is a high-erosion percentage, we should investigate if the shipping dimensions for baseball helmets are calculated correctly in our system.The **Nova Down Jacket** is currently a "break-even" product (Cost = Revenue). While not losing money, it is consuming resources without contributing to net profit.*

---
## 4. Final Recommendations

* Prioritize Platinum Fulfillment: Investigate why Platinum orders are experiencing higher latency. Implementing a "Priority Pick" queue for Platinum members is recommended to correct the current inverted service levels.

* Logistics Optimization: Conduct a deep dive into the Germany (DE) supply chain to identify and remove warehouse bottlenecks.

* Pricing Correction: Audit the 54 "Negative Margin" orders in the Sports Equipment category to identify if the losses are due to shipping miscalculations or excessive discount stacking.

* Inventory Strategy: Re-evaluate the pricing or sourcing cost for the Nova Down Jacket to move it from Break-Even to a positive margin.